# Pan-African Soybean G×E Challenge — Submission Notebook

**Competition:** DataTour 2026 · dataafriquehub.org  
**Task:** Predict soybean yield (kg/ha) for new environments (G×E generalisation)  
**Metric:** RMSE (lower is better)  
**Author:** Sangura J. Nyongesa (ToumölaTech, Guinea)  

## Approach
1. **OOF variety encoding** — mean, median, std, n_trials, CV, Finlay-Wilkinson slope  
2. **OOF location encoding + spatial k-NN** — haversine distance to nearest known sites  
3. **G×E interactions** — variety sensitivity × climate features  
4. **Ensemble** — LightGBM-A (conservative) + LightGBM-B (moderate) + CatBoost  
5. **GroupKFold** on `environment_id` — no leakage, honest OOF score  

All random seeds are fixed → **fully reproducible**.

## 1. Setup & imports

In [ ]:
import os, glob
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility seeds ────────────────────────────────────────────────────
RANDOM_STATE  = 42
N_FOLDS       = 5
K_SPATIAL_NBR = 5
np.random.seed(RANDOM_STATE)

# ── Data paths (auto-detect Kaggle or local) ─────────────────────────────────
_candidates = glob.glob('/kaggle/input/**/train.csv', recursive=True)
if _candidates:
    TRAIN_PATH  = _candidates[0]
    TEST_PATH   = TRAIN_PATH.replace('train.csv', 'test.csv')
    SAMPLE_PATH = TRAIN_PATH.replace('train.csv', 'sample_submission.csv')
else:
    TRAIN_PATH  = 'train.csv'
    TEST_PATH   = 'test.csv'
    SAMPLE_PATH = 'sample_submission.csv'

TARGET  = 'yield_kg_ha'
ID_COL  = 'id'
ENV_COL = 'environment_id'
VAR_COL = 'variety_id'

print('TRAIN_PATH :', TRAIN_PATH)
print('TEST_PATH  :', TEST_PATH)
print('SAMPLE_PATH:', SAMPLE_PATH)

## 2. Load data

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

global_mean = train[TARGET].mean()
gkf = GroupKFold(n_splits=N_FOLDS)

print(f'Train: {train.shape}  |  Test: {test.shape}')
print(f'Train environments: {train[ENV_COL].nunique()}  |  Test environments: {test[ENV_COL].nunique()}')
print(f'Train varieties: {train[VAR_COL].nunique()}  |  Test varieties: {test[VAR_COL].nunique()}')
print(f'Overlap env train/test: {len(set(train[ENV_COL]) & set(test[ENV_COL]))}  (must be 0)')
print(f'Global mean yield: {global_mean:.1f} kg/ha')
print(f'Target std: {train[TARGET].std():.1f} kg/ha')

## 3. Static feature engineering

In [ ]:
def add_static_features(df):
    df = df.copy()
    # Sowing date: cyclic encoding
    dt = pd.to_datetime(df['SOWING'], dayfirst=True, errors='coerce')
    df['sow_doy']     = dt.dt.dayofyear.fillna(180).astype(float)
    df['sow_month']   = dt.dt.month.fillna(6).astype(float)
    df['sow_doy_sin'] = np.sin(2 * np.pi * df['sow_doy'] / 365.25)
    df['sow_doy_cos'] = np.cos(2 * np.pi * df['sow_doy'] / 365.25)
    # Geography
    df['abs_lat']      = df['LAT'].abs()
    df['hemisphere']   = (df['LAT'] >= 0).astype(int)
    df['is_irrigated'] = (~df['RAINFED'].astype(str).str.lower()
                           .str.contains('rainfed', na=False)).astype(int)
    # Climate derived
    b1  = df['wc2.1_30s_bio_1']
    b12 = df['wc2.1_30s_bio_12']
    b5  = df['wc2.1_30s_bio_5']
    b6  = df['wc2.1_30s_bio_6']
    df['temp_range']    = b5 - b6
    df['aridity_proxy'] = b12 / (b1 + 10)
    df['precip_x_temp'] = b12 * b1
    # Soil depth averages + std
    for var in ['clay','sand','silt','nitrogen','phh2o','soc','bdod','cfvo','ocd']:
        cols = [c for c in df.columns if c.startswith(var + '_')]
        if cols:
            df[f'{var}_mean'] = df[cols].mean(axis=1)
            df[f'{var}_std']  = df[cols].std(axis=1).fillna(0)
    return df

train = add_static_features(train)
test  = add_static_features(test)
print('Static features done.')

## 4. OOF variety encoding with stability measures

- **Mean / median / std / n** — basic variety performance
- **CV (coefficient of variation)** — stability: low CV = stable variety
- **Finlay-Wilkinson slope** — sensitivity: β > 1 = responsive (good envs good, bad envs bad)

In [ ]:
def compute_variety_stability(ref_df):
    env_means = ref_df.groupby(ENV_COL)[TARGET].mean().rename('env_mean_ref')
    df2 = ref_df.merge(env_means, on=ENV_COL)

    def fw_slope(g):
        if len(g) < 3:
            return 1.0
        x, y = g['env_mean_ref'].values, g[TARGET].values
        vx = np.var(x)
        return np.cov(x, y)[0, 1] / vx if vx > 0 else 1.0

    stats = (
        ref_df.groupby(VAR_COL)[TARGET]
        .agg(variety_mean='mean', variety_median='median',
             variety_std='std', variety_n='count')
        .assign(variety_cv=lambda d: d['variety_std'] / (d['variety_mean'] + 1))
    )
    fw = df2.groupby(VAR_COL).apply(fw_slope).rename('variety_fw_slope')
    return stats.join(fw)

for col in ['variety_mean','variety_median','variety_std',
            'variety_n','variety_cv','variety_fw_slope']:
    train[col] = np.nan

for _, (fit_idx, val_idx) in enumerate(gkf.split(train, groups=train[ENV_COL])):
    vstats = compute_variety_stability(train.iloc[fit_idx])
    for col in vstats.columns:
        mapped = train.iloc[val_idx][VAR_COL].map(vstats[col])
        train.loc[train.index[val_idx], col] = mapped.values

train['variety_mean']     = train['variety_mean'].fillna(global_mean)
train['variety_median']   = train['variety_median'].fillna(global_mean)
train['variety_std']      = train['variety_std'].fillna(train['variety_std'].median())
train['variety_n']        = train['variety_n'].fillna(1)
train['variety_cv']       = train['variety_cv'].fillna(train['variety_cv'].median())
train['variety_fw_slope'] = train['variety_fw_slope'].fillna(1.0)

full_vstats = compute_variety_stability(train)
test = test.merge(full_vstats, on=VAR_COL, how='left')
test['variety_mean']     = test['variety_mean'].fillna(global_mean)
test['variety_median']   = test['variety_median'].fillna(global_mean)
test['variety_std']      = test['variety_std'].fillna(train['variety_std'].median())
test['variety_n']        = test['variety_n'].fillna(0)
test['variety_cv']       = test['variety_cv'].fillna(train['variety_cv'].median())
test['variety_fw_slope'] = test['variety_fw_slope'].fillna(1.0)

print('Variety features done.')
print('New varieties in test (n=0):', (test['variety_n'] == 0).sum())

## 5. OOF location encoding + spatial k-NN

56% of test sites are absent from train. Spatial k-NN with inverse-distance Haversine weighting
gives a climate proxy for completely new sites.

In [ ]:
train['loc_mean'] = np.nan
train['loc_n']    = np.nan

for _, (fit_idx, val_idx) in enumerate(gkf.split(train, groups=train[ENV_COL])):
    lstats = train.iloc[fit_idx].groupby('loc').agg(
        loc_mean=(TARGET, 'mean'), loc_n=(TARGET, 'count'))
    for col in lstats.columns:
        mapped = train.iloc[val_idx]['loc'].map(lstats[col])
        train.loc[train.index[val_idx], col] = mapped.values

train['loc_mean'] = train['loc_mean'].fillna(global_mean)
train['loc_n']    = train['loc_n'].fillna(0)

full_lstats = train.groupby('loc').agg(
    loc_mean=(TARGET, 'mean'), loc_n=(TARGET, 'count'),
    loc_lat=('LAT', 'first'), loc_lon=('LON', 'first'))
test = test.merge(full_lstats[['loc_mean', 'loc_n']], on='loc', how='left')
test['loc_mean'] = test['loc_mean'].fillna(global_mean)
test['loc_n']    = test['loc_n'].fillna(0)

def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = (np.sin((lat2 - lat1) / 2)**2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2)**2)
    return 2 * 6371 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

known = full_lstats.reset_index()

def spatial_knn_yield(lat, lon, exclude=None):
    d = haversine(lat, lon, known['loc_lat'].values, known['loc_lon'].values)
    if exclude is not None:
        d = np.where(known['loc'].values == exclude, np.inf, d)
    d = np.where(d == 0, 1e-3, d)
    idx = np.argsort(d)[:K_SPATIAL_NBR]
    return np.average(known['loc_mean'].values[idx], weights=1 / d[idx])

train['spatial_knn'] = [spatial_knn_yield(r.LAT, r.LON, r.loc)
                         for r in train.itertuples()]
test['spatial_knn']  = [spatial_knn_yield(r.LAT, r.LON)
                         for r in test.itertuples()]
print('Spatial k-NN done.')

## 6. G×E interaction features

In [ ]:
def add_gxe(df):
    df = df.copy()
    df['gxe_fw_x_precip']  = df['variety_fw_slope'] * df['wc2.1_30s_bio_12']
    df['gxe_fw_x_temp']    = df['variety_fw_slope'] * df['wc2.1_30s_bio_1']
    df['gxe_fw_x_aridity'] = df['variety_fw_slope'] * df['aridity_proxy']
    df['gxe_var_x_knn']    = df['variety_mean'] * df['spatial_knn']
    df['gxe_var_x_loc']    = df['variety_mean'] * df['loc_mean']
    df['knn_x_loc']         = df['spatial_knn'] * df['loc_mean']
    if 'YEAR' in df.columns:
        df['year_x_var'] = df['YEAR'] * df['variety_mean']
    return df

train = add_gxe(train)
test  = add_gxe(test)
print('G×E interaction features done.')

## 7. Feature matrix & label encoding

In [ ]:
STR_COLS = [c for c in ['COUNTRY','SEASON','RAINFED','SOURCE','COMPANY', VAR_COL]
            if c in train.columns and not pd.api.types.is_numeric_dtype(train[c])]
DROP = [ID_COL, TARGET, ENV_COL, 'SOWING', 'loc']
feature_cols = [c for c in train.columns if c not in DROP]

# Label-encode string columns → integers
le_map = {}
for c in STR_COLS:
    if c in feature_cols:
        le = LabelEncoder()
        le.fit(pd.concat([train[c].astype(str), test[c].astype(str)]))
        train[c] = le.transform(train[c].astype(str))
        test[c]  = le.transform(test[c].astype(str))
        le_map[c] = le

# Safety: convert any remaining non-numeric column
for c in feature_cols:
    if not pd.api.types.is_numeric_dtype(train[c]):
        le = LabelEncoder()
        le.fit(pd.concat([train[c].astype(str), test[c].astype(str)]))
        train[c] = le.transform(train[c].astype(str))
        test[c]  = le.transform(test[c].astype(str))

X      = train[feature_cols].astype(np.float32)
y      = train[TARGET].values
X_test = test[feature_cols].astype(np.float32)
groups = train[ENV_COL].values

print(f'{len(feature_cols)} features.')
print('X dtypes unique:', X.dtypes.unique().tolist())

## 8. LightGBM training (GroupKFold on environment_id)

Two configs: A (conservative, mirrors the known-good submission) and B (moderate capacity).

In [ ]:
def run_lgbm(params, tag='LGB'):
    oof   = np.zeros(len(y))
    tpred = np.zeros(len(X_test))
    for fold, (tr_idx, val_idx) in enumerate(
            gkf.split(X, groups=groups)):
        ds_tr = lgb.Dataset(X.iloc[tr_idx], y[tr_idx])
        ds_vl = lgb.Dataset(X.iloc[val_idx], y[val_idx])
        m = lgb.train(
            params, ds_tr, num_boost_round=5000,
            valid_sets=[ds_vl],
            callbacks=[lgb.early_stopping(150, verbose=False),
                       lgb.log_evaluation(0)],
        )
        oof[val_idx] = m.predict(X.iloc[val_idx], num_iteration=m.best_iteration)
        tpred += m.predict(X_test, num_iteration=m.best_iteration) / N_FOLDS
        rmse = np.sqrt(np.mean((y[val_idx] - oof[val_idx])**2))
        print(f'  {tag} fold {fold+1}: RMSE={rmse:.1f}  iter={m.best_iteration}')
    overall = np.sqrt(np.mean((y - oof)**2))
    print(f'  {tag} OOF RMSE: {overall:.1f}')
    return oof, tpred, overall

lgb_params_A = dict(
    objective='regression', metric='rmse', learning_rate=0.02,
    num_leaves=15, max_depth=-1, min_data_in_leaf=50,
    feature_fraction=0.6, bagging_fraction=0.6, bagging_freq=1,
    lambda_l1=1.0, lambda_l2=3.0, verbosity=-1, seed=RANDOM_STATE,
)
lgb_params_B = dict(
    objective='regression', metric='rmse', learning_rate=0.015,
    num_leaves=31, max_depth=-1, min_data_in_leaf=40,
    feature_fraction=0.65, bagging_fraction=0.65, bagging_freq=1,
    lambda_l1=0.5, lambda_l2=2.0, verbosity=-1, seed=RANDOM_STATE+1,
)

print('=== LightGBM A (conservative) ===')
oof_lgbA, pred_lgbA, rmse_A = run_lgbm(lgb_params_A, 'LGBA')
print()
print('=== LightGBM B (moderate) ===')
oof_lgbB, pred_lgbB, rmse_B = run_lgbm(lgb_params_B, 'LGBB')

## 9. CatBoost training

In [ ]:
cb_params = dict(
    iterations=3000, learning_rate=0.02,
    depth=6, l2_leaf_reg=5, bagging_temperature=0.5,
    random_strength=1, min_data_in_leaf=20,
    loss_function='RMSE', eval_metric='RMSE',
    random_seed=RANDOM_STATE, verbose=0,
    early_stopping_rounds=150,
)

oof_cb   = np.zeros(len(y))
pred_cb  = np.zeros(len(X_test))

print('=== CatBoost ===')
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, groups=groups)):
    Xf = X.iloc[tr_idx].values.astype(np.float32)
    yf = y[tr_idx]
    Xv = X.iloc[val_idx].values.astype(np.float32)
    yv = y[val_idx]
    model = CatBoostRegressor(**cb_params)
    model.fit(Xf, yf, eval_set=(Xv, yv), use_best_model=True)
    oof_cb[val_idx] = model.predict(Xv)
    pred_cb += model.predict(X_test.values.astype(np.float32)) / N_FOLDS
    rmse = np.sqrt(np.mean((yv - oof_cb[val_idx])**2))
    print(f'  CB fold {fold+1}: RMSE={rmse:.1f}  iter={model.best_iteration_}')

overall_cb = np.sqrt(np.mean((y - oof_cb)**2))
print(f'  CB OOF RMSE: {overall_cb:.1f}')

## 10. Ensemble weight optimisation & submission

In [ ]:
print('=== Finding optimal ensemble weights ===')
best_rmse = 9e9
best_w    = (0.33, 0.33, 0.34)
for wA in np.arange(0.05, 0.80, 0.05):
    for wB in np.arange(0.05, 0.80, 0.05):
        wC = round(1.0 - wA - wB, 6)
        if wC < 0.05 or wC > 0.90:
            continue
        blend = wA * oof_lgbA + wB * oof_lgbB + wC * oof_cb
        r = np.sqrt(np.mean((y - blend)**2))
        if r < best_rmse:
            best_rmse = r
            best_w = (wA, wB, wC)

wA, wB, wC = best_w
print(f'Weights  LGB-A:{wA:.2f}  LGB-B:{wB:.2f}  CatBoost:{wC:.2f}')
print(f'Ensemble OOF RMSE: {best_rmse:.1f} kg/ha')
print(f'Individual — LGB-A:{rmse_A:.1f}  LGB-B:{rmse_B:.1f}  CB:{overall_cb:.1f}')

# ── Generate submission ──────────────────────────────────────────────────────
test_preds = np.clip(wA * pred_lgbA + wB * pred_lgbB + wC * pred_cb, 0, None)

sample_sub = pd.read_csv(SAMPLE_PATH)
target_col = [c for c in sample_sub.columns if c != ID_COL][0]

sub = sample_sub[[ID_COL]].copy()
sub[target_col] = sub[ID_COL].map(dict(zip(test[ID_COL], test_preds)))

assert sub[target_col].isna().sum() == 0, 'Missing predictions!'
sub.to_csv('submission.csv', index=False)

print(f'\nsubmission.csv saved  ({len(sub)} rows)')
print(sub.head(5).to_string(index=False))
print(f'Preds — min:{test_preds.min():.0f}  mean:{test_preds.mean():.0f}  max:{test_preds.max():.0f}')